# 06 · Mean center against pyproj

**Question.** Does Fieldwork's `mean_center` widget project, average and invert the way it
says it does?

The widget declares an unweighted arithmetic mean of WGS84 UTM easting and northing, with
the centre returned in CRS84. The check transforms the same input points with pyproj,
averages without weights, and compares two things: the projected mean in metres, and the
returned position as a **ground separation** rather than a coordinate difference, because
degrees are not a distance.

Criteria fixed before running: both within 1 mm, and the input count unchanged.


In [ ]:
NEEDED = ["pyproj", "numpy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)


In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_06_mean_center import run

report = run(Path.cwd() / "fixtures" / "mean-center-001.json")
print("criterion:", report["criterion"])
print("agrees:", report["agrees"], "over", report["points"], "points")
print(f"easting  {report['eastingDifferenceM']:.3e} m")
print(f"northing {report['northingDifferenceM']:.3e} m")
print(f"ground separation {report['separationM']:.3e} m")


## What agreement here does not mean

A mean centre is a statistic, and this check says nothing about whether it is the right one.


In [ ]:
# The outlier is the point of the fixture: an unweighted mean is pulled, and both
# implementations are pulled identically. Agreement is not a statement about the statistic.
import pyproj
geod = pyproj.Geod(ellps="WGS84")
fixture = json.loads((Path.cwd() / "fixtures" / "mean-center-001.json").read_text())
centre = report["fieldwork"]["coordinates"]
for point in fixture["input"]["points"]:
    *_ , distance = geod.inv(centre[0], centre[1], point["longitude"], point["latitude"])
    print(f"{point['id']:<4} {distance/1000:7.2f} km from the mean centre")
print()
print(report["notEstablished"])


## Reading the result

**Agrees.** The easting matches exactly, the northing differs by about 9 × 10⁻¹⁰ m, and the
returned position sits about 4 × 10⁻¹⁰ m from the reference — eleven orders of magnitude
inside a 1 mm criterion, and far below any meaningful positional accuracy.

What that establishes is narrow and worth stating precisely: proj4js in the browser and PROJ
in Python agree on this UTM transform, and the averaging is unweighted as declared. What it
does not establish is that the mean centre is a useful summary of these points. The outlier
is 25 km from the cluster and pulls the centre; both implementations are pulled identically,
which is agreement about arithmetic, not about method.
